In [2]:
import os
import sys

print("LLM Wiki 專案初始化成功！")
print("Python 版本：", sys.version.split()[0])

os.makedirs("/content/LLM_Wiki/sources", exist_ok=True)
os.makedirs("/content/LLM_Wiki/wiki", exist_ok=True)
os.makedirs("/content/LLM_Wiki/output", exist_ok=True)

print("專案資料夾建立完成！")

LLM Wiki 專案初始化成功！
Python 版本： 3.13.18
專案資料夾建立完成！


In [3]:

!pip -q install trafilatura beautifulsoup4 requests


In [4]:

import os
import json
import requests
import trafilatura

PROJECT_DIR = "/content/LLM_Wiki"
SOURCE_DIR = f"{PROJECT_DIR}/sources"
os.makedirs(SOURCE_DIR, exist_ok=True)

# 5 篇已選定的官方資料
sources = [
    {
        "id": "SRC-01",
        "title": "Introduction to Large Language Models",
        "topic": "LLM 基礎概念",
        "url": "https://developers.google.com/machine-learning/crash-course/llm"
    },
    {
        "id": "SRC-02",
        "title": "LLMs and Transformers",
        "topic": "Transformer 與 Attention",
        "url": "https://developers.google.com/machine-learning/crash-course/llm/transformers"
    },
    {
        "id": "SRC-03",
        "title": "What Is LLM Training?",
        "topic": "模型訓練",
        "url": "https://www.ibm.com/think/topics/llm-training"
    },
    {
        "id": "SRC-04",
        "title": "Prompt Engineering Best Practices",
        "topic": "Prompt Engineering",
        "url": "https://help.openai.com/en/articles/10032626-prompt-ingineering-best-practices-for-chatgpt"
    },
    {
        "id": "SRC-05",
        "title": "LLM Fundamentals",
        "topic": "LLM 應用與限制",
        "url": "https://learn.microsoft.com/en-us/agent-framework/journey/llm-fundamentals"
    }
]

records = []

for source in sources:
    print(f"\n正在處理：{source['id']} - {source['title']}")

    try:
        response = requests.get(
            source["url"],
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        text = trafilatura.extract(
            response.text,
            include_tables=True,
            include_comments=False
        )

        # 避免把無效或過短內容當成成功 ingest
        if not text or len(text.strip()) < 500:
            raise ValueError("擷取內容不足 500 字元")

        filename = f"{SOURCE_DIR}/{source['id']}.txt"

        with open(filename, "w", encoding="utf-8") as f:
            f.write(text)

        source["status"] = "success"
        source["characters"] = len(text)
        source["file"] = filename

        print(f"✅ 成功，擷取 {len(text)} 個字元")

    except Exception as e:
        source["status"] = "failed"
        source["error"] = str(e)
        print(f"❌ 失敗：{e}")

    records.append(source)

# 保存來源資訊
with open(
    f"{SOURCE_DIR}/metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

success_count = sum(
    r["status"] == "success" for r in records
)

print("\n============================")
print(f"成功匯入：{success_count} / {len(sources)} 篇")
print("============================")

if success_count >= 5:
    print("✅ 已達成至少 5 篇資料的初步匯入目標")
else:
    print("⚠️ 尚未達到 5 篇，需處理失敗來源")



正在處理：SRC-01 - Introduction to Large Language Models
✅ 成功，擷取 5669 個字元

正在處理：SRC-02 - LLMs and Transformers
✅ 成功，擷取 8631 個字元

正在處理：SRC-03 - What Is LLM Training?
✅ 成功，擷取 9430 個字元

正在處理：SRC-04 - Prompt Engineering Best Practices
❌ 失敗：403 Client Error: Forbidden for url: https://help.openai.com/en/articles/10032626-prompt-ingineering-best-practices-for-chatgpt

正在處理：SRC-05 - LLM Fundamentals
✅ 成功，擷取 17361 個字元

成功匯入：4 / 5 篇
⚠️ 尚未達到 5 篇，需處理失敗來源


In [5]:

# 修復 SRC-04：替換為 Microsoft 官方資料
import os
import json
import requests
import trafilatura

SOURCE_DIR = "/content/LLM_Wiki/sources"

new_source = {
    "id": "SRC-04",
    "title": "Prompt Engineering Techniques",
    "topic": "Prompt Engineering",
    "url": "https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/prompt-engineering"
}

try:
    response = requests.get(
        new_source["url"],
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()

    content = trafilatura.extract(
        response.text,
        include_tables=True,
        include_comments=False
    )

    if not content or len(content.strip()) < 500:
        raise ValueError("擷取的內容不足 500 字元")

    filepath = f"{SOURCE_DIR}/SRC-04.txt"

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

    new_source["status"] = "success"
    new_source["characters"] = len(content)
    new_source["file"] = filepath

    print("✅ SRC-04 匯入成功！")
    print("文章：", new_source["title"])
    print("字元數：", len(content))

except Exception as e:
    new_source["status"] = "failed"
    new_source["error"] = str(e)
    print("❌ SRC-04 仍然失敗：", e)

# 同步更新來源紀錄 metadata.json
metadata_path = f"{SOURCE_DIR}/metadata.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    records = json.load(f)

records = [
    new_source if r["id"] == "SRC-04" else r
    for r in records
]

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

success_count = sum(
    r["status"] == "success" for r in records
)

print(f"\n目前成功匯入：{success_count} / 5 篇")


✅ SRC-04 匯入成功！
文章： Prompt Engineering Techniques
字元數： 39281

目前成功匯入：5 / 5 篇


In [6]:

import os
import re
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_DIR = "/content/LLM_Wiki"
SOURCE_DIR = f"{PROJECT_DIR}/sources"
WIKI_DIR = f"{PROJECT_DIR}/wiki"

os.makedirs(WIKI_DIR, exist_ok=True)

# 讀取已匯入的來源資訊
with open(f"{SOURCE_DIR}/metadata.json", "r", encoding="utf-8") as f:
    records = json.load(f)

valid_sources = [
    r for r in records
    if r.get("status") == "success"
    and os.path.isfile(r.get("file", ""))
]

print(f"有效資料來源：{len(valid_sources)} 篇")

if len(valid_sources) < 5:
    raise ValueError("有效來源不足 5 篇，請先檢查資料匯入結果")

# 將文章切成較小段落
def split_text(text, chunk_size=1000, overlap=150):
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    buffer = ""

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # 將較長的段落切開
        while len(para) > chunk_size:
            if buffer:
                chunks.append(buffer)
                buffer = ""

            chunks.append(para[:chunk_size])
            para = para[chunk_size-overlap:]

        if len(buffer) + len(para) + 1 > chunk_size:
            if buffer:
                chunks.append(buffer)
            buffer = para
        else:
            buffer = (buffer + "\n" + para).strip()

    if buffer:
        chunks.append(buffer)

    return chunks

# 建立知識區塊
documents = []

for source in valid_sources:
    with open(source["file"], "r", encoding="utf-8") as f:
        content = f.read()

    chunks = split_text(content)

    for i, chunk in enumerate(chunks):
        documents.append({
            "chunk_id": f"{source['id']}-C{i+1:03d}",
            "source_id": source["id"],
            "title": source["title"],
            "topic": source["topic"],
            "url": source["url"],
            "text": chunk
        })

# 建立 TF-IDF 檢索索引
# 目前資料主要為英文，先用英文問題測試
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

corpus = [
    d["title"] + " " + d["topic"] + " " + d["text"]
    for d in documents
]

tfidf_matrix = vectorizer.fit_transform(corpus)

# 儲存可追溯的資料
with open(f"{WIKI_DIR}/chunks.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("✅ 搜尋索引建立成功！")
print("資料來源數量：", len(valid_sources))
print("知識區塊數量：", len(documents))
print("TF-IDF 矩陣形狀：", tfidf_matrix.shape)


有效資料來源：5 篇
✅ 搜尋索引建立成功！
資料來源數量： 5
知識區塊數量： 96
TF-IDF 矩陣形狀： (96, 8031)


In [7]:

def search_wiki(question, top_k=3):
    """從 LLM Wiki 找出最相關的知識區塊"""

    query_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    top_indices = scores.argsort()[::-1][:top_k]

    results = []

    for idx in top_indices:
        score = float(scores[idx])

        if score <= 0:
            continue

        result = documents[idx].copy()
        result["score"] = score
        results.append(result)

    return results


def ask_wiki(question, top_k=3):
    print("=" * 55)
    print("問題：", question)
    print("=" * 55)

    results = search_wiki(question, top_k)

    if not results:
        print("⚠️ Wiki 找不到相關資料")
        return []

    for i, result in enumerate(results, 1):
        print(f"\n【搜尋結果 {i}】")
        print("來源：", result["title"])
        print("來源 ID：", result["source_id"])
        print("區塊 ID：", result["chunk_id"])
        print("相關分數：", round(result["score"], 4))
        print("網址：", result["url"])
        print("\n內容節錄：")
        print(result["text"][:700])
        print("-" * 55)

    return results


# 第一次搜尋測試
test_results = ask_wiki(
    "What is a large language model?",
    top_k=3
)


問題： What is a large language model?

【搜尋結果 1】
來源： Introduction to Large Language Models
來源 ID： SRC-01
區塊 ID： SRC-01-C007
相關分數： 0.2185
網址： https://developers.google.com/machine-learning/crash-course/llm

內容節錄：
veral sentences.
Although recurrent neural networks learn more context than N-grams, the amount of useful context recurrent neural networks can intuit is still relatively limited. Recurrent neural networks evaluate information "token by token." In contrast, large language models—the topic of the next section—can evaluate the whole context at once.
Note that training recurrent neural networks for long contexts is constrained by the vanishing gradient problem.
Exercise: Check your understanding
- A language model based on 6-grams
- A language model based on 5-grams
-------------------------------------------------------

【搜尋結果 2】
來源： Introduction to Large Language Models
來源 ID： SRC-01
區塊 ID： SRC-01-C002
相關分數： 0.2049
網址： https://developers.google.com/machine-learning/crash-course/llm

In [8]:

wiki_topics = {
    "01_LLM_Basics": {
        "title": "LLM 基本概念",
        "query": "What is a large language model and how does it work?"
    },
    "02_Tokens": {
        "title": "Token 與文字預測",
        "query": "What are tokens and next token prediction?"
    },
    "03_Transformer": {
        "title": "Transformer 與 Attention",
        "query": "How do transformers and self-attention work?"
    },
    "04_Training": {
        "title": "LLM 模型訓練",
        "query": "Pretraining fine-tuning and training language models"
    },
    "05_Applications": {
        "title": "LLM 應用與 Prompt Engineering",
        "query": "Prompt engineering applications examples and instructions"
    },
    "06_Limitations": {
        "title": "LLM 的限制與風險",
        "query": "Hallucination bias limitations and risks of LLMs"
    }
}

for filename, topic in wiki_topics.items():
    results = search_wiki(topic["query"], top_k=4)

    content = f"# {topic['title']}\n\n"
    content += f"**檢索問題：** {topic['query']}\n\n"
    content += "以下是 Wiki 從原始文件檢索出的內容摘錄，尚未進行教學改寫或事實驗證。\n\n"

    for i, result in enumerate(results, 1):
        content += f"## 參考段落 {i}\n\n"
        content += result["text"] + "\n\n"
        content += (
            f"**來源：** [{result['source_id']}] "
            f"{result['title']}  \n"
            f"**區塊：** {result['chunk_id']}  \n"
            f"**網址：** {result['url']}\n\n"
        )

    filepath = f"{WIKI_DIR}/{filename}.md"

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"✅ 已建立：{filename}.md")

print("\nWiki 主題頁面初稿建立完成！")


✅ 已建立：01_LLM_Basics.md
✅ 已建立：02_Tokens.md
✅ 已建立：03_Transformer.md
✅ 已建立：04_Training.md
✅ 已建立：05_Applications.md
✅ 已建立：06_Limitations.md

Wiki 主題頁面初稿建立完成！


In [9]:

import os

wiki_files = sorted([
    f for f in os.listdir("/content/LLM_Wiki/wiki")
    if f.endswith(".md")
])

print("Wiki 主題頁面數量：", len(wiki_files))

for file in wiki_files:
    print("✅", file)

print("\n檢索測試：")
results = search_wiki(
    "What is Transformer attention?",
    top_k=2
)

for result in results:
    print(
        result["source_id"],
        result["chunk_id"],
        round(result["score"], 4)
    )


Wiki 主題頁面數量： 6
✅ 01_LLM_Basics.md
✅ 02_Tokens.md
✅ 03_Transformer.md
✅ 04_Training.md
✅ 05_Applications.md
✅ 06_Limitations.md

檢索測試：
SRC-02 SRC-02-C003 0.1571
SRC-02 SRC-02-C006 0.1563


In [10]:

import os
import json

LOG_DIR = "/content/LLM_Wiki/logs"
os.makedirs(LOG_DIR, exist_ok=True)

# 每個主題準備不同的查詢方式
query_variants = {
    "LLM 基本概念": [
        "What is a large language model?",
        "Definition and capabilities of large language models",
        "Introduction to LLM and language modeling"
    ],
    "Token 與文字預測": [
        "What are tokens and next token prediction?",
        "Tokenization in large language models",
        "How language models predict tokens"
    ],
    "Transformer 與 Attention": [
        "How does transformer self attention work?",
        "Transformer architecture and attention mechanism",
        "Self attention in language models"
    ],
    "LLM 模型訓練": [
        "How are large language models trained?",
        "Pretraining and fine tuning language models",
        "LLM training process and parameters"
    ],
    "LLM 應用與 Prompt Engineering": [
        "Prompt engineering and LLM applications",
        "How to write effective prompts",
        "Applications and use cases of large language models"
    ],
    "LLM 的限制與風險": [
        "LLM hallucinations and limitations",
        "Bias and risks of language models",
        "Hallucinations safety and limitations of LLMs"
    ]
}

# 查詢與檢查 Loop
def wiki_retrieval_loop(topic, max_attempts=3):

    questions = query_variants[topic]
    history = []

    for attempt, question in enumerate(
        questions[:max_attempts], start=1
    ):

        print(f"\n第 {attempt} 次查詢")
        print("問題：", question)

        results = search_wiki(question, top_k=3)

        # 檢查是否有足夠相關的結果
        valid_results = [
            r for r in results
            if r["score"] >= 0.05
            and len(r["text"].strip()) >= 100
        ]

        passed = len(valid_results) >= 2

        history.append({
            "attempt": attempt,
            "query": question,
            "total_results": len(results),
            "valid_results": len(valid_results),
            "passed": passed
        })

        if passed:
            print("✅ 檢索通過")
            print("有效結果：", len(valid_results))

            return {
                "topic": topic,
                "status": "passed",
                "results": valid_results,
                "history": history
            }

        print("⚠️ 檢索未通過，重新查詢...")

    print("❌ 已達最大查詢次數")

    return {
        "topic": topic,
        "status": "failed",
        "results": [],
        "history": history
    }


# 測試六個主題
workflow_results = []

for topic in query_variants:
    print("\n" + "=" * 55)
    print("主題：", topic)

    result = wiki_retrieval_loop(topic)
    workflow_results.append(result)

# 儲存執行紀錄
with open(
    f"{LOG_DIR}/retrieval_loop.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        workflow_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n============================")
print("Loop 測試完成")
print("============================")

for result in workflow_results:
    print(
        result["topic"],
        "→",
        result["status"],
        "| 查詢次數：",
        len(result["history"])
    )



主題： LLM 基本概念

第 1 次查詢
問題： What is a large language model?
✅ 檢索通過
有效結果： 3

主題： Token 與文字預測

第 1 次查詢
問題： What are tokens and next token prediction?
✅ 檢索通過
有效結果： 3

主題： Transformer 與 Attention

第 1 次查詢
問題： How does transformer self attention work?
✅ 檢索通過
有效結果： 3

主題： LLM 模型訓練

第 1 次查詢
問題： How are large language models trained?
✅ 檢索通過
有效結果： 3

主題： LLM 應用與 Prompt Engineering

第 1 次查詢
問題： Prompt engineering and LLM applications
✅ 檢索通過
有效結果： 3

主題： LLM 的限制與風險

第 1 次查詢
問題： LLM hallucinations and limitations
✅ 檢索通過
有效結果： 3

Loop 測試完成
LLM 基本概念 → passed | 查詢次數： 1
Token 與文字預測 → passed | 查詢次數： 1
Transformer 與 Attention → passed | 查詢次數： 1
LLM 模型訓練 → passed | 查詢次數： 1
LLM 應用與 Prompt Engineering → passed | 查詢次數： 1
LLM 的限制與風險 → passed | 查詢次數： 1


In [11]:

import json
import os

LOG_DIR = "/content/LLM_Wiki/logs"

def inspect_retrieval_results(workflow_results):
    report = []

    for workflow in workflow_results:
        topic = workflow["topic"]
        results = workflow["results"]

        seen_ids = set()
        issues = []

        for result in results:
            chunk_id = result.get("chunk_id")
            source_id = result.get("source_id")
            url = result.get("url")
            content = result.get("text", "")

            if not chunk_id or not source_id:
                issues.append("缺少知識區塊或來源 ID")

            if not url or not url.startswith("https://"):
                issues.append("缺少有效格式的來源網址")

            if chunk_id in seen_ids:
                issues.append(f"重複區塊：{chunk_id}")

            seen_ids.add(chunk_id)

            if len(content.strip()) < 100:
                issues.append(f"內容過短：{chunk_id}")

        report.append({
            "topic": topic,
            "status": "needs_review" if issues else "format_passed",
            "issues": issues,
            "result_count": len(results)
        })

    return report


quality_report = inspect_retrieval_results(workflow_results)

for item in quality_report:
    print("=" * 45)
    print("主題：", item["topic"])
    print("格式檢查：", item["status"])
    print("檢索區塊：", item["result_count"])

    if item["issues"]:
        for issue in item["issues"]:
            print("⚠️", issue)
    else:
        print("✅ 來源欄位、區塊及長度檢查通過")

with open(
    f"{LOG_DIR}/quality_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(quality_report, f, ensure_ascii=False, indent=2)


主題： LLM 基本概念
格式檢查： format_passed
檢索區塊： 3
✅ 來源欄位、區塊及長度檢查通過
主題： Token 與文字預測
格式檢查： format_passed
檢索區塊： 3
✅ 來源欄位、區塊及長度檢查通過
主題： Transformer 與 Attention
格式檢查： format_passed
檢索區塊： 3
✅ 來源欄位、區塊及長度檢查通過
主題： LLM 模型訓練
格式檢查： format_passed
檢索區塊： 3
✅ 來源欄位、區塊及長度檢查通過
主題： LLM 應用與 Prompt Engineering
格式檢查： format_passed
檢索區塊： 3
✅ 來源欄位、區塊及長度檢查通過
主題： LLM 的限制與風險
格式檢查： format_passed
檢索區塊： 3
✅ 來源欄位、區塊及長度檢查通過


In [12]:

# 列出六個主題的第一筆檢索內容
# 用來人工確認內容是否符合教學主題

for workflow in workflow_results:

    print("\n" + "=" * 60)
    print("主題：", workflow["topic"])

    if not workflow["results"]:
        print("❌ 沒有可檢查的內容")
        continue

    result = workflow["results"][0]

    print("來源：", result["title"])
    print("來源 ID：", result["source_id"])
    print("相關分數：", round(result["score"], 4))
    print("原始網址：", result["url"])

    print("\n檢索內容：")
    print(result["text"][:1000])



主題： LLM 基本概念
來源： Introduction to Large Language Models
來源 ID： SRC-01
相關分數： 0.2185
原始網址： https://developers.google.com/machine-learning/crash-course/llm

檢索內容：
veral sentences.
Although recurrent neural networks learn more context than N-grams, the amount of useful context recurrent neural networks can intuit is still relatively limited. Recurrent neural networks evaluate information "token by token." In contrast, large language models—the topic of the next section—can evaluate the whole context at once.
Note that training recurrent neural networks for long contexts is constrained by the vanishing gradient problem.
Exercise: Check your understanding
- A language model based on 6-grams
- A language model based on 5-grams

主題： Token 與文字預測
來源： Introduction to Large Language Models
來源 ID： SRC-01
相關分數： 0.1737
原始網址： https://developers.google.com/machine-learning/crash-course/llm

檢索內容：
r the emergence of large language models that evaluate the whole context simultaneously.
What is a language

In [13]:

import re
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SOURCE_DIR = "/content/LLM_Wiki/sources"
WIKI_DIR = "/content/LLM_Wiki/wiki"
LOG_DIR = "/content/LLM_Wiki/logs"

# 1. 依照句子切分，盡量避免切斷英文句子
def split_sentences(text, max_chars=1000):
    sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if len(current) + len(sentence) > max_chars and current:
            chunks.append(current)
            current = ""

        # 超長句子保留完整內容，不強行切斷
        current = (current + " " + sentence).strip()

    if current:
        chunks.append(current)

    return chunks


# 2. 重新建立知識區塊
with open(f"{SOURCE_DIR}/metadata.json", encoding="utf-8") as f:
    sources = json.load(f)

new_documents = []

for src in sources:
    if src.get("status") != "success":
        continue

    filepath = src.get("file")
    if not filepath or not os.path.isfile(filepath):
        continue

    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    for i, chunk in enumerate(split_sentences(content)):
        new_documents.append({
            "chunk_id": f"{src['id']}-C{i+1:03d}",
            "source_id": src["id"],
            "title": src["title"],
            "url": src["url"],
            "text": chunk
        })

# 3. 重新建立搜尋索引
new_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

new_matrix = new_vectorizer.fit_transform([
    d["text"] for d in new_documents
])

# 4. 為每個主題設定優先來源與必要概念
topic_rules = {
    "LLM 基本概念": {
        "sources": ["SRC-01", "SRC-05"],
        "keywords": ["large language model", "language model"],
        "queries": [
            "What is a large language model?",
            "Large language model definition",
            "What are LLMs?"
        ]
    },
    "Token 與文字預測": {
        "sources": ["SRC-01", "SRC-05"],
        "keywords": ["token", "prediction"],
        "queries": [
            "What is a token in language models?",
            "Token prediction in LLMs",
            "Tokenization and next token prediction"
        ]
    },
    "Transformer 與 Attention": {
        "sources": ["SRC-02"],
        "keywords": ["self-attention", "transformer"],
        "queries": [
            "What is self-attention?",
            "How do transformers use attention?",
            "Transformer architecture"
        ]
    },
    "LLM 模型訓練": {
        "sources": ["SRC-03"],
        "keywords": ["training", "pretraining", "fine-tuning"],
        "queries": [
            "How are large language models trained?",
            "LLM pretraining process",
            "Fine-tuning large language models"
        ]
    },
    "LLM 應用與 Prompt Engineering": {
        "sources": ["SRC-04", "SRC-05"],
        "keywords": ["prompt", "instruction", "example"],
        "queries": [
            "How to write effective prompts?",
            "Prompt engineering techniques and examples",
            "LLM applications and instructions"
        ]
    },
    "LLM 的限制與風險": {
        "sources": ["SRC-05"],
        "keywords": ["hallucination", "limitations", "bias"],
        "queries": [
            "What are LLM limitations?",
            "Hallucinations in language models",
            "Risks and bias of LLMs"
        ]
    }
}

# 5. 具備重新查詢功能的檢索
def improved_wiki_search(topic):
    rule = topic_rules[topic]
    history = []

    for attempt, query in enumerate(rule["queries"], 1):

        query_vec = new_vectorizer.transform([query])
        scores = cosine_similarity(query_vec, new_matrix)[0]

        candidates = []

        for i, doc in enumerate(new_documents):
            if doc["source_id"] not in rule["sources"]:
                continue

            text_lower = doc["text"].lower()

            keyword_found = any(
                k in text_lower for k in rule["keywords"]
            )

            if (
                scores[i] >= 0.05
                and keyword_found
                and len(doc["text"].strip()) >= 100
            ):
                result = doc.copy()
                result["score"] = float(scores[i])
                candidates.append(result)

        candidates.sort(
            key=lambda x: x["score"],
            reverse=True
        )

        history.append({
            "attempt": attempt,
            "query": query,
            "matches": len(candidates)
        })

        if candidates:
            return {
                "topic": topic,
                "status": "needs_manual_review",
                "results": candidates[:3],
                "history": history
            }

    return {
        "topic": topic,
        "status": "failed",
        "results": [],
        "history": history
    }


# 6. 執行六個主題
improved_results = []

for topic in topic_rules:
    result = improved_wiki_search(topic)
    improved_results.append(result)

    print("\n" + "=" * 55)
    print("主題：", topic)
    print("狀態：", result["status"])
    print("查詢次數：", len(result["history"]))

    for r in result["results"][:2]:
        print("\n來源：", r["title"])
        print("相關分數：", round(r["score"], 4))
        print("內容：", r["text"][:350])

# 7. 儲存新的測試紀錄
os.makedirs(LOG_DIR, exist_ok=True)

with open(
    f"{LOG_DIR}/improved_retrieval.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        improved_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✅ 改良版檢索測試已完成")



主題： LLM 基本概念
狀態： needs_manual_review
查詢次數： 1

來源： Introduction to Large Language Models
相關分數： 0.1762
內容： Similarly, the punchline of a long joke makes you laugh because you can remember the context from the joke's setup. In language models, context is helpful information before or after the target token. Context can help a language model determine whether "orange" refers to a citrus fruit or a color. Context can help a language model make better predi

來源： LLM Fundamentals
相關分數： 0.161
內容： Note Access to this page requires authorization. You can try signing in or changing directories. Access to this page requires authorization. You can try changing directories. Before building AI agents, it helps to understand the technology that powers them: large language models (LLMs). This page gives you a developer-oriented overview of what LLMs

主題： Token 與文字預測
狀態： needs_manual_review
查詢次數： 1

來源： Introduction to Large Language Models
相關分數： 0.1948
內容： Page Summary - This module explores language 

In [14]:

import os
import json
import re

WIKI_DIR = "/content/LLM_Wiki/wiki"
LOG_DIR = "/content/LLM_Wiki/logs"

# 每個主題需要的重要概念
review_rules = {
    "LLM 基本概念": {
        "sources": ["SRC-01", "SRC-05"],
        "terms": ["large language model", "language models"],
        "required": ["language model"]
    },
    "Token 與文字預測": {
        "sources": ["SRC-01"],
        "terms": ["token", "probability", "prediction"],
        "required": ["token"]
    },
    "Transformer 與 Attention": {
        "sources": ["SRC-02"],
        "terms": ["self-attention", "transformer", "context"],
        "required": ["self-attention"]
    },
    "LLM 模型訓練": {
        "sources": ["SRC-03"],
        "terms": ["training", "pre-training", "fine-tuning"],
        "required": ["training"]
    },
    "LLM 應用與 Prompt Engineering": {
        "sources": ["SRC-04", "SRC-05"],
        "terms": ["prompt", "instruction", "example", "task"],
        "required": ["prompt", "instruction"]
    },
    "LLM 的限制與風險": {
        "sources": ["SRC-05"],
        "terms": ["hallucination", "limitations", "bias"],
        "required": ["hallucination", "limitations"]
    }
}

# 排除不適合當作教學內容的文字
bad_phrases = [
    "access to this page requires authorization",
    "you can try signing in",
    "you can try changing directories"
]

def review_candidates(topic):
    rule = review_rules[topic]
    candidates = []

    for doc in new_documents:
        if doc["source_id"] not in rule["sources"]:
            continue

        text_lower = doc["text"].lower()

        if any(bad in text_lower for bad in bad_phrases):
            continue

        if not any(term in text_lower for term in rule["required"]):
            continue

        if len(doc["text"].strip()) < 150:
            continue

        # 以核心概念的涵蓋數量排序
        score = sum(
            1 for term in rule["terms"]
            if term in text_lower
        )

        candidate = doc.copy()
        candidate["concept_score"] = score
        candidates.append(candidate)

    candidates.sort(
        key=lambda x: x["concept_score"],
        reverse=True
    )

    return candidates[:3]


# 重新檢查六大主題
reviewed_wiki = {}

for topic in review_rules:
    results = review_candidates(topic)
    reviewed_wiki[topic] = results

    print("\n" + "=" * 55)
    print("主題：", topic)
    print("候選段落數：", len(results))

    if results:
        best = results[0]
        print("來源：", best["title"])
        print("概念涵蓋分數：", best["concept_score"])
        print("內容預覽：")
        print(best["text"][:500])
    else:
        print("⚠️ 沒有符合條件的內容，需要重新檢索")

# 儲存人工審查候選資料
os.makedirs(LOG_DIR, exist_ok=True)

with open(
    f"{WIKI_DIR}/reviewed_materials.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reviewed_wiki,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✅ 教學素材初步篩選完成")
print("檔案：reviewed_materials.json")



主題： LLM 基本概念
候選段落數： 3
來源： Introduction to Large Language Models
概念涵蓋分數： 2
內容預覽：
Page Summary - This module explores language models, which estimate the probability of a token or sequence of tokens occurring within a longer sequence, enabling tasks like text generation, translation, and summarization. - Language models utilize context, the surrounding information of a target token, to enhance prediction accuracy, with recurrent neural networks offering more context than traditional N-grams. - N-grams are ordered sequences of words used to build language models, with longer N

主題： Token 與文字預測
候選段落數： 3
來源： Introduction to Large Language Models
概念涵蓋分數： 3
內容預覽：
Page Summary - This module explores language models, which estimate the probability of a token or sequence of tokens occurring within a longer sequence, enabling tasks like text generation, translation, and summarization. - Language models utilize context, the surrounding information of a target token, to enhance prediction accuracy

In [15]:

import os
import json

WIKI_DIR = "/content/LLM_Wiki/wiki"
os.makedirs(WIKI_DIR, exist_ok=True)

# 讀取原始來源資訊
with open(
    "/content/LLM_Wiki/sources/metadata.json",
    "r",
    encoding="utf-8"
) as f:
    source_records = json.load(f)

source_map = {
    r["id"]: r for r in source_records
    if r.get("status") == "success"
}

# 根據已檢索的來源，整理繁體中文教學初稿
wiki_pages = [
    {
        "filename": "01_LLM_Basics_ZH.md",
        "title": "什麼是大型語言模型？",
        "source_ids": ["SRC-01"],
        "definition": (
            "大型語言模型（Large Language Model，LLM）"
            "是一種透過大量資料訓練的語言模型，"
            "能根據輸入的上下文產生文字。"
        ),
        "explanation": (
            "語言模型會估計不同文字單位出現的機率。"
            "大型語言模型可以運用上下文資訊，"
            "完成問答、摘要、翻譯及文字生成等任務。"
        ),
        "example": (
            "例如，當使用者輸入「請幫我整理這篇文章」，"
            "LLM 可以根據提供的文章內容產生摘要。"
        ),
        "terms": ["LLM", "Language Model", "Context"]
    },
    {
        "filename": "02_Tokens_ZH.md",
        "title": "Token 與文字預測",
        "source_ids": ["SRC-01"],
        "definition": (
            "Token 是語言模型處理文字時使用的基本單位，"
            "可能是一個字、單字、部分單字或其他文字片段。"
        ),
        "explanation": (
            "語言模型會利用上下文估計下一個 Token "
            "出現的可能性，再逐步產生文字。"
            "實際切分方式會依 Tokenizer 而不同。"
        ),
        "example": (
            "例如，輸入「今天的天氣很」，"
            "模型可能接續產生「好」、「冷」等文字。"
            "這是文字預測的簡化示意。"
        ),
        "terms": ["Token", "Tokenizer", "Prediction"]
    },
    {
        "filename": "03_Transformer_ZH.md",
        "title": "Transformer 與 Attention",
        "source_ids": ["SRC-02"],
        "definition": (
            "Transformer 是現代大型語言模型"
            "常使用的一種神經網路架構。"
        ),
        "explanation": (
            "Self-Attention（自注意力）機制"
            "可以協助模型衡量不同 Token 之間的關聯，"
            "進而利用上下文處理語言資訊。"
        ),
        "example": (
            "例如「小明把書交給小華，因為他要離開」，"
            "模型需要參考句子中的其他詞語，"
            "判斷「他」可能指的是誰。"
            "此例用於說明上下文關聯，並非保證能正確判斷。"
        ),
        "terms": ["Transformer", "Attention", "Self-Attention"]
    },
    {
        "filename": "04_Training_ZH.md",
        "title": "大型語言模型如何訓練？",
        "source_ids": ["SRC-03"],
        "definition": (
            "LLM 訓練是透過資料與最佳化演算法，"
            "逐步調整模型參數的過程。"
        ),
        "explanation": (
            "預訓練（Pre-training）讓模型學習廣泛的語言模式。"
            "微調（Fine-tuning）則可使用特定資料，"
            "讓模型更適合某些任務。"
            "部分模型還會經過其他後訓練程序。"
        ),
        "example": (
            "例如，模型先透過一般文字資料進行預訓練，"
            "之後再透過特定領域資料微調，"
            "使其更適合處理該領域的任務。"
        ),
        "terms": ["Pre-training", "Fine-tuning", "Parameters"]
    },
    {
        "filename": "05_Prompt_ZH.md",
        "title": "LLM 應用與 Prompt Engineering",
        "source_ids": ["SRC-04", "SRC-05"],
        "definition": (
            "Prompt Engineering（提示工程）"
            "是設計與改善輸入指令的方法，"
            "目的是讓模型更清楚理解任務需求。"
        ),
        "explanation": (
            "良好的 Prompt 可以提供明確的任務、"
            "必要背景、限制條件與期望的輸出格式。"
            "LLM 常被應用於文字摘要、翻譯與問答等工作。"
        ),
        "example": (
            "較模糊的提問：「幫我整理文章。」"
            "較清楚的提問：「請將這篇文章整理成"
            "三個重點，使用繁體中文，"
            "每點不超過50字。」"
        ),
        "terms": ["Prompt", "Instruction", "Context"]
    },
    {
        "filename": "06_Limitations_ZH.md",
        "title": "LLM 的限制與風險",
        "source_ids": ["SRC-05"],
        "definition": (
            "大型語言模型雖然能產生流暢文字，"
            "但生成結果不一定正確。"
        ),
        "explanation": (
            "常見限制包括幻覺（Hallucination）、"
            "資訊可能過時，以及回答內容需要查證。"
            "模型可能產生看似合理但實際錯誤的資訊。"
        ),
        "example": (
            "例如，詢問 AI 某篇論文的作者與出版年份時，"
            "模型可能提供錯誤資訊。"
            "應回到論文原始資料確認。"
        ),
        "terms": ["Hallucination", "Limitations", "Fact-checking"]
    }
]

# 寫入 Wiki Markdown 頁面
for page in wiki_pages:
    content = f"# {page['title']}\n\n"
    content += "> 狀態：教學初稿，待人工確認\n\n"

    content += "## 一、基本定義\n\n"
    content += page["definition"] + "\n\n"

    content += "## 二、原理說明\n\n"
    content += page["explanation"] + "\n\n"

    content += "## 三、生活化例子\n\n"
    content += page["example"] + "\n\n"

    content += "## 四、重要英文術語\n\n"
    for term in page["terms"]:
        content += f"- {term}\n"

    content += "\n## 五、資料來源\n\n"
    for sid in page["source_ids"]:
        src = source_map[sid]
        content += (
            f"- [{sid}] [{src['title']}]({src['url']})\n"
        )

    # 附上真正被篩選出的知識區塊 ID
    content += "\n## 六、Wiki 檢索依據\n\n"
    for result in reviewed_wiki.get(
        page["title"].replace("什麼是大型語言模型？", "LLM 基本概念"),
        []
    ):
        content += f"- {result['chunk_id']}\n"

    filepath = os.path.join(WIKI_DIR, page["filename"])

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

    print("✅ 已建立：", page["filename"])

print("\n繁體中文 Wiki 教學初稿建立完成！")


✅ 已建立： 01_LLM_Basics_ZH.md
✅ 已建立： 02_Tokens_ZH.md
✅ 已建立： 03_Transformer_ZH.md
✅ 已建立： 04_Training_ZH.md
✅ 已建立： 05_Prompt_ZH.md
✅ 已建立： 06_Limitations_ZH.md

繁體中文 Wiki 教學初稿建立完成！


In [16]:

import os

WIKI_DIR = "/content/LLM_Wiki/wiki"

zh_files = sorted([
    name for name in os.listdir(WIKI_DIR)
    if name.endswith("_ZH.md")
])

print("繁體中文 Wiki 頁面數量：", len(zh_files))

for filename in zh_files:
    print("✅", filename)

# 預覽第一篇 Wiki
if zh_files:
    with open(
        os.path.join(WIKI_DIR, zh_files[0]),
        "r",
        encoding="utf-8"
    ) as f:
        print("\n第一篇 Wiki 內容預覽：\n")
        print(f.read()[:1500])


繁體中文 Wiki 頁面數量： 6
✅ 01_LLM_Basics_ZH.md
✅ 02_Tokens_ZH.md
✅ 03_Transformer_ZH.md
✅ 04_Training_ZH.md
✅ 05_Prompt_ZH.md
✅ 06_Limitations_ZH.md

第一篇 Wiki 內容預覽：

# 什麼是大型語言模型？

> 狀態：教學初稿，待人工確認

## 一、基本定義

大型語言模型（Large Language Model，LLM）是一種透過大量資料訓練的語言模型，能根據輸入的上下文產生文字。

## 二、原理說明

語言模型會估計不同文字單位出現的機率。大型語言模型可以運用上下文資訊，完成問答、摘要、翻譯及文字生成等任務。

## 三、生活化例子

例如，當使用者輸入「請幫我整理這篇文章」，LLM 可以根據提供的文章內容產生摘要。

## 四、重要英文術語

- LLM
- Language Model
- Context

## 五、資料來源

- [SRC-01] [Introduction to Large Language Models](https://developers.google.com/machine-learning/crash-course/llm)

## 六、Wiki 檢索依據

- SRC-01-C001
- SRC-01-C006
- SRC-05-C006



In [17]:

import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

WIKI_DIR = "/content/LLM_Wiki/wiki"

# 讀取六篇繁體中文 Wiki
wiki_documents = []

for filename in sorted(os.listdir(WIKI_DIR)):
    if not filename.endswith("_ZH.md"):
        continue

    filepath = os.path.join(WIKI_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    title_match = re.search(r"^# (.+)$", content, re.MULTILINE)
    title = title_match.group(1) if title_match else filename

    wiki_documents.append({
        "filename": filename,
        "title": title,
        "content": content
    })

# 使用字元 n-gram 支援繁體中文檢索
zh_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    sublinear_tf=True
)

zh_matrix = zh_vectorizer.fit_transform([
    doc["title"] + "\n" + doc["content"]
    for doc in wiki_documents
])

print("✅ 中文 Wiki 搜尋索引建立成功")
print("Wiki 頁面數量：", len(wiki_documents))


def search_zh_wiki(question, top_k=3):
    query_vector = zh_vectorizer.transform([question])
    scores = cosine_similarity(query_vector, zh_matrix)[0]

    ranked_indices = scores.argsort()[::-1][:top_k]

    results = []

    for idx in ranked_indices:
        if scores[idx] <= 0:
            continue

        doc = wiki_documents[idx].copy()
        doc["score"] = float(scores[idx])
        results.append(doc)

    return results


def ask_zh_wiki(question, top_k=3):
    results = search_zh_wiki(question, top_k)

    print("\n問題：", question)
    print("=" * 55)

    if not results:
        print("找不到相關 Wiki 內容")
        return []

    for i, result in enumerate(results, 1):
        print(f"\n【結果 {i}】")
        print("Wiki 頁面：", result["title"])
        print("相關分數：", round(result["score"], 4))
        print("檔案：", result["filename"])
        print("\n內容預覽：")
        print(result["content"][:800])
        print("-" * 55)

    return results


# 第一次中文查詢測試
ask_zh_wiki("什麼是大型語言模型？", top_k=2)


✅ 中文 Wiki 搜尋索引建立成功
Wiki 頁面數量： 6

問題： 什麼是大型語言模型？

【結果 1】
Wiki 頁面： 什麼是大型語言模型？
相關分數： 0.2593
檔案： 01_LLM_Basics_ZH.md

內容預覽：
# 什麼是大型語言模型？

> 狀態：教學初稿，待人工確認

## 一、基本定義

大型語言模型（Large Language Model，LLM）是一種透過大量資料訓練的語言模型，能根據輸入的上下文產生文字。

## 二、原理說明

語言模型會估計不同文字單位出現的機率。大型語言模型可以運用上下文資訊，完成問答、摘要、翻譯及文字生成等任務。

## 三、生活化例子

例如，當使用者輸入「請幫我整理這篇文章」，LLM 可以根據提供的文章內容產生摘要。

## 四、重要英文術語

- LLM
- Language Model
- Context

## 五、資料來源

- [SRC-01] [Introduction to Large Language Models](https://developers.google.com/machine-learning/crash-course/llm)

## 六、Wiki 檢索依據

- SRC-01-C001
- SRC-01-C006
- SRC-05-C006

-------------------------------------------------------

【結果 2】
Wiki 頁面： 大型語言模型如何訓練？
相關分數： 0.0577
檔案： 04_Training_ZH.md

內容預覽：
# 大型語言模型如何訓練？

> 狀態：教學初稿，待人工確認

## 一、基本定義

LLM 訓練是透過資料與最佳化演算法，逐步調整模型參數的過程。

## 二、原理說明

預訓練（Pre-training）讓模型學習廣泛的語言模式。微調（Fine-tuning）則可使用特定資料，讓模型更適合某些任務。部分模型還會經過其他後訓練程序。

## 三、生活化例子

例如，模型先透過一般文字資料進行預訓練，之後再透過特定領域資料微調，使其更適合處理該領域的任務。

## 四、重要英文術語

- Pre-training
- Fine-tuning
- Parameters

##

[{'filename': '01_LLM_Basics_ZH.md',
  'title': '什麼是大型語言模型？',
  'content': '# 什麼是大型語言模型？\n\n> 狀態：教學初稿，待人工確認\n\n## 一、基本定義\n\n大型語言模型（Large Language Model，LLM）是一種透過大量資料訓練的語言模型，能根據輸入的上下文產生文字。\n\n## 二、原理說明\n\n語言模型會估計不同文字單位出現的機率。大型語言模型可以運用上下文資訊，完成問答、摘要、翻譯及文字生成等任務。\n\n## 三、生活化例子\n\n例如，當使用者輸入「請幫我整理這篇文章」，LLM 可以根據提供的文章內容產生摘要。\n\n## 四、重要英文術語\n\n- LLM\n- Language Model\n- Context\n\n## 五、資料來源\n\n- [SRC-01] [Introduction to Large Language Models](https://developers.google.com/machine-learning/crash-course/llm)\n\n## 六、Wiki 檢索依據\n\n- SRC-01-C001\n- SRC-01-C006\n- SRC-05-C006\n',
  'score': 0.259288845901952},
 {'filename': '04_Training_ZH.md',
  'title': '大型語言模型如何訓練？',
  'content': '# 大型語言模型如何訓練？\n\n> 狀態：教學初稿，待人工確認\n\n## 一、基本定義\n\nLLM 訓練是透過資料與最佳化演算法，逐步調整模型參數的過程。\n\n## 二、原理說明\n\n預訓練（Pre-training）讓模型學習廣泛的語言模式。微調（Fine-tuning）則可使用特定資料，讓模型更適合某些任務。部分模型還會經過其他後訓練程序。\n\n## 三、生活化例子\n\n例如，模型先透過一般文字資料進行預訓練，之後再透過特定領域資料微調，使其更適合處理該領域的任務。\n\n## 四、重要英文術語\n\n- Pre-training\n- Fine-tuning\n- Parameters\n\n## 五、資料來源\n\

In [19]:

test_questions = [
    "什麼是 Token？",
    "Transformer 的 Attention 是什麼？",
    "大型語言模型如何訓練？",
    "怎麼寫出好的 Prompt？",
    "LLM 為什麼會產生幻覺？"
]

for question in test_questions:
    results = search_zh_wiki(question, top_k=1)

    print("\n問題：", question)

    if results:
        print("最相關 Wiki：", results[0]["title"])
        print("相關分數：", round(results[0]["score"], 4))
    else:
        print("❌ 沒有搜尋結果")



問題： 什麼是 Token？
最相關 Wiki： Token 與文字預測
相關分數： 0.2201

問題： Transformer 的 Attention 是什麼？
最相關 Wiki： Transformer 與 Attention
相關分數： 0.4963

問題： 大型語言模型如何訓練？
最相關 Wiki： 大型語言模型如何訓練？
相關分數： 0.2569

問題： 怎麼寫出好的 Prompt？
最相關 Wiki： LLM 應用與 Prompt Engineering
相關分數： 0.2572

問題： LLM 為什麼會產生幻覺？
最相關 Wiki： LLM 的限制與風險
相關分數： 0.0929


In [20]:

import os
import json
import re

WIKI_DIR = "/content/LLM_Wiki/wiki"
LOG_DIR = "/content/LLM_Wiki/logs"

# 中文 Wiki 與檢索主題的對應表
page_topic_map = {
    "01_LLM_Basics_ZH.md": "LLM 基本概念",
    "02_Tokens_ZH.md": "Token 與文字預測",
    "03_Transformer_ZH.md": "Transformer 與 Attention",
    "04_Training_ZH.md": "LLM 模型訓練",
    "05_Prompt_ZH.md": "LLM 應用與 Prompt Engineering",
    "06_Limitations_ZH.md": "LLM 的限制與風險"
}

# 依原本教學規劃設定允許的來源
page_sources = {
    "01_LLM_Basics_ZH.md": ["SRC-01"],
    "02_Tokens_ZH.md": ["SRC-01"],
    "03_Transformer_ZH.md": ["SRC-02"],
    "04_Training_ZH.md": ["SRC-03"],
    "05_Prompt_ZH.md": ["SRC-04", "SRC-05"],
    "06_Limitations_ZH.md": ["SRC-05"]
}

audit_report = []

for filename, topic in page_topic_map.items():

    filepath = os.path.join(WIKI_DIR, filename)

    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    # 保留原本的中文教學內容
    content = re.split(
        r"\n## 五、資料來源",
        content
    )[0].rstrip()

    # 重建來源章節
    content += "\n\n## 五、資料來源\n\n"

    for sid in page_sources[filename]:
        src = source_map[sid]
        content += (
            f"- [{sid}] "
            f"[{src['title']}]({src['url']})\n"
        )

    # 取得實際篩選的候選區塊
    candidates = reviewed_wiki.get(topic, [])

    # 只保留屬於該頁允許來源的區塊
    valid_chunks = [
        r for r in candidates
        if r["source_id"] in page_sources[filename]
    ]

    content += "\n## 六、Wiki 檢索依據\n\n"

    if valid_chunks:
        for r in valid_chunks:
            content += (
                f"- {r['chunk_id']} "
                f"（{r['source_id']}）\n"
            )
    else:
        content += "- 尚無符合來源條件的候選區塊\n"

    content += (
        "\n> 注意：以上為檢索候選依據，"
        "尚需人工核對是否支持各段中文敘述。\n"
    )

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

    audit_report.append({
        "filename": filename,
        "topic": topic,
        "source_ids": page_sources[filename],
        "valid_chunk_count": len(valid_chunks),
        "status": (
            "pending_manual_review"
            if valid_chunks else "missing_evidence"
        )
    })

# 儲存檢查結果
os.makedirs(LOG_DIR, exist_ok=True)

with open(
    f"{LOG_DIR}/wiki_citation_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("====== Wiki 引用檢查結果 ======")

for item in audit_report:
    print(
        item["topic"],
        "| 候選區塊：",
        item["valid_chunk_count"],
        "|",
        item["status"]
    )

print("\n✅ Wiki 引用整理完成")


====== Wiki 引用檢查結果 ======
LLM 基本概念 | 候選區塊： 2 | pending_manual_review
Token 與文字預測 | 候選區塊： 3 | pending_manual_review
Transformer 與 Attention | 候選區塊： 3 | pending_manual_review
LLM 模型訓練 | 候選區塊： 3 | pending_manual_review
LLM 應用與 Prompt Engineering | 候選區塊： 3 | pending_manual_review
LLM 的限制與風險 | 候選區塊： 2 | pending_manual_review

✅ Wiki 引用整理完成


In [21]:

import os
import re

WIKI_DIR = "/content/LLM_Wiki/wiki"

for filename, topic in page_topic_map.items():
    filepath = os.path.join(WIKI_DIR, filename)

    with open(filepath, encoding="utf-8") as f:
        content = f.read()

    # 顯示中文教學內容，略過後面的引用清單
    main_content = re.split(
        r"\n## 五、資料來源",
        content
    )[0]

    print("\n" + "=" * 65)
    print("Wiki 主題：", topic)
    print("=" * 65)
    print(main_content)

    print("\n【對應英文檢索依據】")

    allowed = set(page_sources[filename])
    candidates = [
        r for r in reviewed_wiki.get(topic, [])
        if r["source_id"] in allowed
    ]

    if not candidates:
        print("⚠️ 尚無符合來源條件的候選區塊")

    for i, result in enumerate(candidates, 1):
        print(f"\n參考區塊 {i}")
        print("來源：", result["source_id"])
        print("區塊：", result["chunk_id"])
        print("英文內容：")
        print(result["text"])

print("\n✅ 六個主題的內容與引用依據已列出")



Wiki 主題： LLM 基本概念
# 什麼是大型語言模型？

> 狀態：教學初稿，待人工確認

## 一、基本定義

大型語言模型（Large Language Model，LLM）是一種透過大量資料訓練的語言模型，能根據輸入的上下文產生文字。

## 二、原理說明

語言模型會估計不同文字單位出現的機率。大型語言模型可以運用上下文資訊，完成問答、摘要、翻譯及文字生成等任務。

## 三、生活化例子

例如，當使用者輸入「請幫我整理這篇文章」，LLM 可以根據提供的文章內容產生摘要。

## 四、重要英文術語

- LLM
- Language Model
- Context


【對應英文檢索依據】

參考區塊 1
來源： SRC-01
區塊： SRC-01-C001
英文內容：
Page Summary - This module explores language models, which estimate the probability of a token or sequence of tokens occurring within a longer sequence, enabling tasks like text generation, translation, and summarization. - Language models utilize context, the surrounding information of a target token, to enhance prediction accuracy, with recurrent neural networks offering more context than traditional N-grams. - N-grams are ordered sequences of words used to build language models, with longer N-grams providing more context but potentially encountering sparsity issues. - Tokens, the atomic units of language modeling, represent words, subwords, 

In [22]:

import os
import re
import json
import html
from pathlib import Path
from IPython.display import display, HTML

WIKI_DIR = Path("/content/LLM_Wiki/wiki")
OUTPUT_DIR = Path("/content/LLM_Wiki/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 讀取真正建立的六篇 Wiki
files = [
    "01_LLM_Basics_ZH.md",
    "02_Tokens_ZH.md",
    "03_Transformer_ZH.md",
    "04_Training_ZH.md",
    "05_Prompt_ZH.md",
    "06_Limitations_ZH.md"
]

pages = []

for filename in files:
    path = WIKI_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"找不到 Wiki：{filename}")

    raw = path.read_text(encoding="utf-8")

    def section(name):
        pattern = rf"## {re.escape(name)}\s*\n(.*?)(?=\n## |\Z)"
        match = re.search(pattern, raw, re.S)
        return match.group(1).strip() if match else ""

    pages.append({
        "filename": filename,
        "title": raw.splitlines()[0].replace("# ", ""),
        "definition": section("一、基本定義"),
        "explanation": section("二、原理說明"),
        "example": section("三、生活化例子"),
        "terms": section("四、重要英文術語"),
        "sources": section("五、資料來源")
    })

print("✅ 成功讀取 Wiki：", len(pages), "篇")

# 2. HTML 內容輔助函式
def esc(text):
    return html.escape(str(text))

def paragraph(text):
    return f'<p>{esc(text)}</p>'

def topic_card(page, field="definition"):
    return f"""
    <div class="content-card">
        <div class="section-label">{esc(page["title"])}</div>
        {paragraph(page[field])}
    </div>
    """

def source_links(page):
    links = re.findall(
        r'\[([^\]]+)\]\((https?://[^)]+)\)',
        page["sources"]
    )
    return "".join(
        f'<a href="{esc(url)}" target="_blank" '
        f'rel="noopener noreferrer">{esc(name)}</a>'
        for name, url in links
    )

def make_slide(title, subtitle, content, source_page=None):
    citation = ""
    if source_page is not None:
        citation = (
            '<div class="citation">Wiki 資料來源：'
            + source_links(source_page)
            + '</div>'
        )
    return {
        "title": title,
        "subtitle": subtitle,
        "content": content + citation
    }

# 3. 建立十二頁教學簡報
slides = []

slides.append(make_slide(
    "大型語言模型 LLM 入門",
    "從基本概念、運作原理到實際應用",
    """
    <div class="hero">
      <div class="hero-icon">AI</div>
      <h2>認識生成式 AI 背後的語言技術</h2>
      <p>利用 LLM Wiki 知識庫製作的互動式教學簡報</p>
    </div>
    """
))

slides.append(make_slide(
    "什麼是 LLM？",
    "01 / 基本概念",
    topic_card(pages[0]) +
    '<h3>生活化例子</h3>' +
    paragraph(pages[0]["example"]),
    pages[0]
))

slides.append(make_slide(
    "什麼是 Token？",
    "02 / 文字的處理單位",
    topic_card(pages[1]) +
    '<h3>為什麼需要 Token？</h3>' +
    paragraph(pages[1]["explanation"]),
    pages[1]
))

slides.append(make_slide(
    "LLM 如何預測文字？",
    "03 / 互動示範",
    paragraph(pages[1]["explanation"]) +
    """
    <div class="content-card">
      <h3>今天的天氣很……</h3>
      <p id="prediction">點擊按鈕，觀看文字預測示例。</p>
      <button onclick="predictText()">產生示例文字</button>
      <p class="small">此為預先設定的教學示範，
      並非真實模型即時運算。</p>
    </div>
    """,
    pages[1]
))

slides.append(make_slide(
    "Transformer 與 Attention",
    "04 / 模型如何利用上下文",
    topic_card(pages[2], "definition") +
    '<h3>Self-Attention 是什麼？</h3>' +
    paragraph(pages[2]["explanation"]) +
    '<h3>生活化例子</h3>' +
    paragraph(pages[2]["example"]),
    pages[2]
))

slides.append(make_slide(
    "LLM 如何訓練？",
    "05 / 從資料到模型",
    """
    <div class="steps">
      <div><strong>① 預訓練</strong><span>學習廣泛的語言模式</span></div>
      <div><strong>② 微調</strong><span>適應特定任務或領域</span></div>
      <div><strong>③ 後訓練</strong><span>依需求調整模型行為</span></div>
    </div>
    """ +
    paragraph(pages[3]["explanation"]) +
    paragraph(pages[3]["example"]),
    pages[3]
))

slides.append(make_slide(
    "LLM 的常見應用",
    "06 / 生活與學習",
    """
    <div class="tiles">
      <div>📝<strong>文字摘要</strong><span>整理文章重點</span></div>
      <div>🌐<strong>語言翻譯</strong><span>轉換不同語言</span></div>
      <div>💬<strong>問答輔助</strong><span>回答使用者問題</span></div>
      <div>📋<strong>文字處理</strong><span>整理與改寫內容</span></div>
    </div>
    """ +
    paragraph(pages[4]["explanation"]),
    pages[4]
))

slides.append(make_slide(
    "Prompt Engineering",
    "07 / 如何讓 AI 理解你的需求",
    topic_card(pages[4], "definition") +
    '<h3>好的 Prompt 有什麼不同？</h3>' +
    paragraph(pages[4]["example"]) +
    '<div class="tip">提示：清楚描述任務、背景、限制與輸出格式。</div>',
    pages[4]
))

slides.append(make_slide(
    "LLM 的限制與風險",
    "08 / 為什麼需要查證？",
    topic_card(pages[5], "definition") +
    paragraph(pages[5]["explanation"]) +
    '<h3>實際例子</h3>' +
    paragraph(pages[5]["example"]),
    pages[5]
))

slides.append(make_slide(
    "互動知識測驗",
    "09 / 檢查你的學習成果",
    """
    <div class="content-card">
      <h3 id="quiz-question"></h3>
      <div id="quiz-options"></div>
      <p id="quiz-feedback"></p>
      <button onclick="nextQuestion()">下一題</button>
    </div>
    """
))

slides.append(make_slide(
    "課程重點整理",
    "10 / 今天學到了什麼？",
    """
    <div class="tiles">
      <div><strong>LLM</strong><span>根據上下文產生文字</span></div>
      <div><strong>Token</strong><span>模型處理文字的單位</span></div>
      <div><strong>Transformer</strong><span>利用注意力處理上下文</span></div>
      <div><strong>Prompt</strong><span>清楚表達任務需求</span></div>
    </div>
    <div class="tip">LLM 很實用，但重要資訊仍需要查證。</div>
    """
))

references = ""
seen_urls = set()

for p in pages:
    for name, url in re.findall(
        r'\[([^\]]+)\]\((https?://[^)]+)\)',
        p["sources"]
    ):
        if url not in seen_urls:
            seen_urls.add(url)
            references += (
                f'<p><a href="{esc(url)}" target="_blank" '
                f'rel="noopener noreferrer">{esc(name)}</a></p>'
            )

slides.append(make_slide(
    "參考資料",
    "11 / LLM Wiki 的知識來源",
    '<div class="content-card">' +
    references +
    '</div><p class="small">本簡報由六篇繁體中文 Wiki '
    '教學初稿整理而成，並保留原始文章連結。</p>'
))

assert len(slides) == 12

# 4. 儲存簡報資料，方便追蹤 Wiki 到簡報的轉換
data_path = OUTPUT_DIR / "slides_data.json"
data_path.write_text(
    json.dumps(slides, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("✅ 已建立 12 頁教學內容")


✅ 成功讀取 Wiki： 6 篇
✅ 已建立 12 頁教學內容


In [23]:

from pathlib import Path
from IPython.display import display, HTML

OUTPUT_DIR = Path("/content/LLM_Wiki/output")

html_template = r"""
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>大型語言模型 LLM 入門</title>

<style>
* { box-sizing: border-box; }

body {
    margin: 0;
    font-family: "Microsoft JhengHei", "Noto Sans TC", sans-serif;
    background: #0b1020;
    color: #f3f4ff;
}

header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 18px 5%;
    color: #b8bde0;
    font-size: 14px;
}

.logo {
    color: #b7a7ff;
    font-weight: bold;
    letter-spacing: 2px;
}

.progress {
    height: 5px;
    background: #252945;
}

#progress-fill {
    height: 100%;
    width: 0%;
    background: linear-gradient(90deg, #8b5cf6, #38bdf8);
    transition: width .3s;
}

main {
    max-width: 1050px;
    min-height: 70vh;
    margin: 35px auto 10px;
    padding: 40px 55px;
    border: 1px solid #303653;
    border-radius: 24px;
    background: linear-gradient(145deg, #171e35, #101629);
    box-shadow: 0 20px 60px #0005;
}

.kicker {
    color: #a99aff;
    font-size: 14px;
    letter-spacing: 2px;
}

h1 {
    font-size: clamp(29px, 4vw, 48px);
    margin: 20px 0;
    line-height: 1.3;
}

h2 { font-size: 25px; }
h3 { color: #c8bbff; margin-top: 30px; }

p {
    font-size: 18px;
    line-height: 1.9;
    color: #dde2f4;
}

.content-card {
    background: #212a46;
    padding: 24px;
    margin: 22px 0;
    border-radius: 16px;
    border: 1px solid #394362;
}

.section-label {
    color: #a99aff;
    font-size: 15px;
    font-weight: bold;
}

.hero { text-align: center; padding: 45px 0; }

.hero-icon {
    display: grid;
    place-items: center;
    width: 120px;
    height: 120px;
    border-radius: 30px;
    background: linear-gradient(135deg, #7648e9, #269aca);
    font-size: 50px;
    font-weight: bold;
    margin: auto;
}

.tiles {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 16px;
    margin: 25px 0;
}

.tiles > div {
    background: #212a46;
    padding: 25px;
    border-radius: 15px;
    font-size: 25px;
}

.tiles strong, .tiles span, .steps span {
    display: block;
    margin-top: 12px;
}

.tiles strong { font-size: 20px; }
.tiles span, .steps span {
    color: #bec8de;
    font-size: 16px;
}

.steps {
    display: grid;
    gap: 12px;
}

.steps > div {
    padding: 18px;
    background: #212a46;
    border-left: 4px solid #a78bfa;
    border-radius: 10px;
}

.tip {
    background: #30274d;
    padding: 20px;
    border-radius: 12px;
    margin-top: 24px;
    line-height: 1.8;
}

.citation {
    border-top: 1px solid #394362;
    margin-top: 30px;
    padding-top: 15px;
    color: #b8c2df;
    font-size: 13px;
}

a {
    color: #97d5ff;
    overflow-wrap: anywhere;
    margin-right: 10px;
}

.small { color: #aab5d1; font-size: 13px; }

button {
    background: #7655dc;
    border: 0;
    border-radius: 10px;
    color: white;
    padding: 13px 22px;
    cursor: pointer;
    font-size: 16px;
    margin: 5px;
}

button:hover { background: #9474ff; }
button:disabled { opacity: .35; cursor: default; }

.controls {
    display: flex;
    justify-content: center;
    gap: 15px;
    padding: 25px;
}

.option {
    display: block;
    width: 100%;
    text-align: left;
    background: #333e60;
    margin: 10px 0;
}

#quiz-feedback { color: #c9b7ff; }

@media(max-width: 700px) {
    main {
        padding: 25px 20px;
        margin: 12px;
    }
    .tiles { grid-template-columns: 1fr; }
    p { font-size: 16px; }
    header { padding: 15px; }
}
</style>
</head>

<body>
<header>
    <div class="logo">LLM BEGINNER WIKI</div>
    <div id="counter">1 / 12</div>
</header>

<div class="progress">
    <div id="progress-fill"></div>
</div>

<main>
    <div id="subtitle" class="kicker"></div>
    <h1 id="title"></h1>
    <div id="slide-content"></div>
</main>

<div class="controls">
    <button id="prev" onclick="changeSlide(-1)">上一頁</button>
    <button id="next" onclick="changeSlide(1)">下一頁</button>
</div>

<script>
const slides = __SLIDES_DATA__;
let current = 0;

function render() {
    const slide = slides[current];
    document.getElementById("title").textContent = slide.title;
    document.getElementById("subtitle").textContent = slide.subtitle;
    document.getElementById("slide-content").innerHTML = slide.content;

    document.getElementById("counter").textContent =
        (current + 1) + " / " + slides.length;

    document.getElementById("progress-fill").style.width =
        ((current + 1) / slides.length * 100) + "%";

    document.getElementById("prev").disabled = current === 0;
    document.getElementById("next").disabled =
        current === slides.length - 1;

    if (current === 9) showQuestion();
}

function changeSlide(step) {
    current = Math.max(0, Math.min(slides.length - 1, current + step));
    render();
    window.scrollTo(0, 0);
}

document.addEventListener("keydown", function(e) {
    if (e.key === "ArrowRight") changeSlide(1);
    if (e.key === "ArrowLeft") changeSlide(-1);
});

const predictions = ["好", "冷", "熱", "舒服"];
let predictionIndex = 0;

function predictText() {
    const word = predictions[predictionIndex % predictions.length];
    predictionIndex++;
    document.getElementById("prediction").textContent =
        "今天的天氣很" + word + "。";
}

const questions = [
    {
        q: "LLM 主要是根據什麼產生文字？",
        options: ["上下文與學到的模式", "隨機查詢網頁", "固定答案表"],
        answer: 0,
        why: "LLM 會依據上下文及學到的模式預測 Token。"
    },
    {
        q: "Transformer 中的 Self-Attention 主要有何作用？",
        options: ["控制螢幕亮度", "衡量 Token 之間的關聯", "刪除所有文字"],
        answer: 1,
        why: "Self-Attention 協助模型利用上下文資訊。"
    },
    {
        q: "使用 LLM 時，哪種做法比較適當？",
        options: ["所有回答都直接相信", "不需要檢查來源", "重要資訊應另外查證"],
        answer: 2,
        why: "模型可能產生幻覺或過時資訊。"
    }
];

let quizIndex = 0;

function showQuestion() {
    const item = questions[quizIndex];
    document.getElementById("quiz-question").textContent =
        (quizIndex + 1) + ". " + item.q;

    document.getElementById("quiz-feedback").textContent = "";

    document.getElementById("quiz-options").innerHTML =
        item.options.map((option, i) =>
            '<button class="option" onclick="answerQuestion(' + i + ')">' +
            option + '</button>'
        ).join("");
}

function answerQuestion(index) {
    const item = questions[quizIndex];
    document.getElementById("quiz-feedback").textContent =
        (index === item.answer ? "✅ 答對了！ " : "❌ 再複習一下。 ") +
        item.why;
}

function nextQuestion() {
    quizIndex = (quizIndex + 1) % questions.length;
    showQuestion();
}

render();
</script>
</body>
</html>
"""

# 將已產生的 Wiki 簡報資料放入 HTML
slides_json = json.dumps(slides, ensure_ascii=False)
slides_json = slides_json.replace("</", "<\\/")

final_html = html_template.replace(
    "__SLIDES_DATA__",
    slides_json
)

output_path = OUTPUT_DIR / "LLM_入門教學簡報.html"
output_path.write_text(final_html, encoding="utf-8")

print("✅ HTML 簡報產生成功！")
print("頁數：", len(slides))
print("檔案位置：", output_path)
print("檔案大小：", round(output_path.stat().st_size / 1024, 1), "KB")


✅ HTML 簡報產生成功！
頁數： 12
檔案位置： /content/LLM_Wiki/output/LLM_入門教學簡報.html
檔案大小： 16.0 KB


In [24]:

from google.colab import files

files.download(
    "/content/LLM_Wiki/output/LLM_入門教學簡報.html"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:

import shutil
from google.colab import files

# 將六篇 Wiki 打包成 ZIP
shutil.make_archive(
    "/content/LLM_Wiki_中文知識庫",
    "zip",
    "/content/LLM_Wiki/wiki"
)

files.download("/content/LLM_Wiki_中文知識庫.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>